# Car Sales Live OpenAI + vLLM Demo

This notebook runs the `CarSales` environment as a true online alternating game:

- the buyer questions are generated live by `gpt-4.1-mini`
- the seller replies are generated live by a local `vllm` model
- each buyer question conditions on the immediately previous seller response
- each seller response gets a deception label inside the environment

Use a kernel with `openai` and `vllm` available, plus a valid `OPENAI_API_KEY`.


In [ ]:
import os
os.environ.setdefault('VLLM_WORKER_MULTIPROC_METHOD', 'spawn')

import importlib
import json
import re
import sys
from pathlib import Path
from pprint import pprint

from openai import OpenAI
import torch
from vllm import LLM, SamplingParams

ROOT = Path('/playpen-ssd/smerrill/deception2')
CAR_ROOT = ROOT / 'CarSales'
CAR_SRC = CAR_ROOT / 'src'
if str(CAR_SRC) not in sys.path:
    sys.path.insert(0, str(CAR_SRC))

import car_sales_environment as car_sales_env
importlib.reload(car_sales_env)

UsedCarSalesEnvironment = car_sales_env.UsedCarSalesEnvironment
CarSalesSpec = car_sales_env.CarSalesSpec


In [ ]:
BUYER_MODEL_NAME = 'gpt-4.1-mini'
SELLER_MODEL_NAME = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'
SCENARIO_NAME = 'ford_f150_xlt'
SEED = 0
MAX_ROUNDS = 4

OPENAI_BASE_URL = None
OPENAI_TIMEOUT = 120.0
BUYER_TEMPERATURE = 0.4
BUYER_TOP_P = 1.0
BUYER_MAX_OUTPUT_TOKENS = 220

DTYPE = 'bfloat16'
MAX_MODEL_LEN = 8192
GPU_MEMORY_UTILIZATION = 0.9
TRUST_REMOTE_CODE = False
TENSOR_PARALLEL_SIZE = max(1, torch.cuda.device_count())
SELLER_TEMPERATURE = 0.3
SELLER_TOP_P = 0.95
SELLER_MAX_TOKENS = 320

print('buyer_model =', BUYER_MODEL_NAME)
print('seller_model =', SELLER_MODEL_NAME)
print('scenario_name =', SCENARIO_NAME)
print('cuda_device_count =', torch.cuda.device_count())


In [ ]:
def maybe_parse_json_object(value):
    if isinstance(value, dict):
        return value
    if value is None:
        return None

    text = str(value).strip()
    if not text:
        return None

    candidates = [text]
    match = re.search(r'\{.*\}', text, flags=re.S)
    if match:
        candidates.append(match.group(0))

    for candidate in candidates:
        try:
            parsed = json.loads(candidate)
        except Exception:
            continue
        if isinstance(parsed, dict):
            return parsed
    return None


def extract_responses_output_text(response):
    text = getattr(response, 'output_text', None)
    if text:
        return text
    output = getattr(response, 'output', []) or []
    parts = []
    for item in output:
        for content in getattr(item, 'content', []) or []:
            chunk_text = getattr(content, 'text', None)
            if chunk_text:
                parts.append(chunk_text)
    return ''.join(parts).strip()


def show_messages(messages):
    for idx, msg in enumerate(messages):
        print(f'===== MESSAGE {idx} / role={msg["role"]} =====')
        print(msg['content'])
        print()


def show_dialogue(dialogue_history):
    for item in dialogue_history:
        extras = []
        if item.get('question_focus'):
            extras.append(f"focus={item['question_focus']}")
        if item.get('disclosed_issue_ids'):
            extras.append(f"disclosed={item['disclosed_issue_ids']}")
        suffix = '' if not extras else ' [' + '; '.join(extras) + ']'
        print(f"{item['speaker']}: {item['message']}{suffix}")
        print()


def show_agent_calls(agent, show_prompt=False):
    for idx, call in enumerate(agent.call_log):
        print(f'===== CALL {idx} / agent={agent.name} =====')
        print('model =', call.get('model_name'))
        print('raw_text =', call.get('raw_text'))
        if show_prompt:
            print('prompt =')
            for msg in call.get('messages', []):
                print(f"[{msg['role']}] {msg['content']}")
                print()


In [ ]:
class OpenAIBuyerAgent:
    def __init__(
        self,
        name,
        client,
        model_name,
        reasoning_instruction='COD',
        instruction_format='reasoning',
        temperature=0.4,
        top_p=1.0,
        max_output_tokens=220,
    ):
        self.name = name
        self.client = client
        self.model_name = model_name
        self.reasoning_instruction = reasoning_instruction
        self.instruction_format = instruction_format
        self.temperature = float(temperature)
        self.top_p = float(top_p)
        self.max_output_tokens = int(max_output_tokens)
        self.call_log = []

    def chat(self, messages, num_responses=1, debug=False):
        response = self.client.responses.create(
            model=self.model_name,
            input=[
                {
                    'role': msg.get('role', 'user'),
                    'content': [{'type': 'input_text', 'text': str(msg.get('content', ''))}],
                }
                for msg in messages
            ],
            temperature=self.temperature,
            top_p=self.top_p,
            max_output_tokens=self.max_output_tokens,
        )
        raw_text = extract_responses_output_text(response)
        parsed = maybe_parse_json_object(raw_text)
        self.call_log.append({
            'model_name': self.model_name,
            'messages': messages,
            'raw_text': raw_text,
            'parsed': parsed,
        })
        return [parsed if parsed is not None else raw_text]


In [ ]:
class VLLMSellerAgent:
    def __init__(
        self,
        name,
        model,
        model_name,
        base_seed=0,
        reasoning_instruction='COD',
        instruction_format='reasoning',
        temperature=0.3,
        top_p=0.95,
        max_tokens=320,
    ):
        self.name = name
        self.model = model
        self.model_name = model_name
        self.base_seed = int(base_seed)
        self.reasoning_instruction = reasoning_instruction
        self.instruction_format = instruction_format
        self.temperature = float(temperature)
        self.top_p = float(top_p)
        self.max_tokens = int(max_tokens)
        self.call_log = []

    def chat(self, messages, num_responses=1, debug=False):
        sampling_params = SamplingParams(
            temperature=self.temperature,
            top_p=self.top_p,
            max_tokens=self.max_tokens,
            seed=self.base_seed + len(self.call_log),
        )
        outputs = self.model.chat(messages, sampling_params, use_tqdm=False)
        raw_text = outputs[0].outputs[0].text
        parsed = maybe_parse_json_object(raw_text)
        self.call_log.append({
            'model_name': self.model_name,
            'messages': messages,
            'raw_text': raw_text,
            'parsed': parsed,
        })
        return [parsed if parsed is not None else raw_text]


In [ ]:
api_key = os.getenv('OPENAI_API_KEY')
if OPENAI_BASE_URL and not api_key:
    api_key = 'EMPTY'
if not api_key:
    raise ValueError('Set OPENAI_API_KEY before running this notebook.')

openai_client = OpenAI(api_key=api_key, base_url=OPENAI_BASE_URL, timeout=OPENAI_TIMEOUT)
llm = LLM(
    model=SELLER_MODEL_NAME,
    dtype=DTYPE,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    trust_remote_code=TRUST_REMOTE_CODE,
    seed=SEED,
)


In [ ]:
def make_env(seed=SEED, scenario_name=SCENARIO_NAME, max_rounds=MAX_ROUNDS):
    seller = VLLMSellerAgent(
        name='Seller',
        model=llm,
        model_name=SELLER_MODEL_NAME,
        base_seed=seed,
        reasoning_instruction='COD',
        instruction_format='reasoning',
        temperature=SELLER_TEMPERATURE,
        top_p=SELLER_TOP_P,
        max_tokens=SELLER_MAX_TOKENS,
    )
    buyer = OpenAIBuyerAgent(
        name='Buyer',
        client=openai_client,
        model_name=BUYER_MODEL_NAME,
        reasoning_instruction='COD',
        instruction_format='reasoning',
        temperature=BUYER_TEMPERATURE,
        top_p=BUYER_TOP_P,
        max_output_tokens=BUYER_MAX_OUTPUT_TOKENS,
    )
    env = UsedCarSalesEnvironment(
        agents=[seller, buyer],
        seed=seed,
        scenario_name=scenario_name,
        spec=CarSalesSpec(max_rounds=max_rounds),
        history_window=12,
    )
    return env, seller, buyer


In [ ]:
env, seller_agent, buyer_agent = make_env()
state = env.get_state(include_system_prompt=True)

print('phase =', state['phase'])
print('scenario =', state['scenario']['name'])
print('seller_known_issue_ids =', state['hidden_state']['seller_known_issue_ids'])
print('\n=== INITIAL BUYER PROMPT ===\n')
show_messages(state['messages'])


In [ ]:
first_result = env.step()
print('=== handled phase:', first_result['phase'], '===')
print('action =')
pprint(first_result['action'])
if first_result.get('label') is not None:
    print('label =')
    pprint(first_result['label'])
print()

print('=== SELLER PROMPT AFTER FIRST BUYER QUESTION ===')
print()
show_messages(env.get_state(include_system_prompt=True)['messages'])
print()

while not env.game_over():
    result = env.step()
    print('=== handled phase:', result['phase'], '===')
    print('action =')
    pprint(result['action'])
    if result.get('label') is not None:
        print('label =')
        pprint(result['label'])
    print()

print('=== FINAL DIALOGUE ===')
print()
show_dialogue(env.dialogue_history)


In [ ]:
print('=== BUYER CALLS ===')
show_agent_calls(buyer_agent, show_prompt=False)

print('=== SELLER CALLS ===')
show_agent_calls(seller_agent, show_prompt=False)
